### Install Library yang digunakan

Uncomment cell berikut kemudian run 

In [1]:
# ! pip install selenium bs4 pandas tqdm rapidfuzz geopandas

### Import library

In [2]:
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from bs4 import BeautifulSoup
from tqdm import tqdm
from rapidfuzz import fuzz
import urllib.parse
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import re
import time

### Function Utility

In [3]:
def extract_lat_lng(url):
    match = re.search(r'/@(-?\d+\.\d+),(-?\d+\.\d+)', url)
    if match:
        return float(match.group(1)), float(match.group(2))
    return None, None


def wait_for_url_coordinates(driver, max_attempts=10):
    for _ in range(max_attempts):
        url = driver.current_url
        if '/@' in url and re.search(r'/@-?\d+\.\d+,-?\d+\.\d+', url):
            return url
        time.sleep(1)
    return driver.current_url


def wait_page_change(driver, timeout=10):
    initial = driver.page_source
    wait = WebDriverWait(driver, timeout)
    wait.until(lambda d: d.page_source != initial)


def get_google_name(soup):
    h1 = soup.find('h1', class_='DUwDvf lfPIob')
    return h1.get_text(strip=True) if h1 else None

def is_google_blocked(driver):
    src = driver.page_source.lower()
    return (
        "detected unusual traffic" in src
        or "sorry/index" in driver.current_url
        or "recaptcha" in src
    )

### Main Scrapper

In [4]:
def get_long_lat(driver, location, nmdesa, max_retry=3):

    search_query = f"{location} {nmdesa}"
    encoded = urllib.parse.quote(search_query)
    url_search = f"https://www.google.com/maps/search/{encoded}"

    for attempt in range(max_retry):

        driver.get(url_search)
        driver.implicitly_wait(3)

        wait_page_change(driver)

        # detect captcha / block
        if is_google_blocked(driver):
            print("Google block detected.")
            return [None, None, None, None, 3, None]

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # CASE 1: Multiple results
        if soup.find('h1', class_="fontTitleLarge IFMGgb"):

            link = soup.find('a', class_='hfpxzc')
            if not link:
                return [None, None, None, None, 1, None]

            driver.get(link.get("href"))
            wait_page_change(driver)

            url = wait_for_url_coordinates(driver)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            nama_google = get_google_name(soup)

            lat, lng = extract_lat_lng(url)

            if lat and lng:
                temp_closed = int("temporarily closed" in driver.page_source.lower())
                return [lat, lng, nama_google, url, 0, temp_closed]

            return [None, None, nama_google, url, 1, None]

        # CASE 2: Direct result
        if soup.find('h1', class_='DUwDvf lfPIob'):

            url = wait_for_url_coordinates(driver)
            nama_google = get_google_name(soup)

            lat, lng = extract_lat_lng(url)

            if lat and lng:
                temp_closed = int("temporarily closed" in driver.page_source.lower())
                return [lat, lng, nama_google, url, 0, temp_closed]

            return [None, None, nama_google, url, 1, None]

        # CASE 3: Not found
        return [None, None, None, None, 2, None]

    # failed after retries
    return [None, None, None, None, 3, None]

## DATA PREPARATION

In [5]:
gdf_boundary = gpd.read_file('input/jatinegara-boundary.gpkg')

### Ganti sesuai dengan pembagian tugas

In [6]:
row_start = 20771
row_end = 24924

In [7]:
file_input = f"Data-SBR-{row_start}-{row_end}.csv"

df = pd.read_csv(f"proses/{file_input}", sep=";", dtype=object)

for col in ["latitude", "longitude", "nama_google", "url", "isna", "temporary_closed", "similarity", "inside_boundary"]:
    if col not in df.columns:
        df[col] = ""

df_na = df[df["latitude"].isna() | (df["latitude"] == "")]
df_no_na = df[~(df["latitude"].isna() | (df["latitude"] == ""))]

# Create column to check similarity name
df_no_na["similarity"] = df_no_na.apply(
    lambda row: fuzz.token_set_ratio(
        str(row["nama"]).lower(),
        str(row["nama_google"]).lower()
    ),
    axis=1
)

# Check if df_no_na is inside boundary geometry
df_no_na = df_no_na.copy()
gdf_point = gpd.GeoDataFrame(
    df_no_na,
    geometry=gpd.points_from_xy(df_no_na["longitude"].astype(float),
                                df_no_na["latitude"].astype(float)),
    crs="EPSG:4326"
)

gdf_point = gdf_point.to_crs(gdf_boundary.crs)
polygon = gdf_boundary.geometry.union_all()
gdf_point["inside_boundary"] = gdf_point.within(polygon).astype(int)
df_no_na["inside_boundary"] = gdf_point["inside_boundary"]

df_na["isna"] = df_na["isna"].apply(lambda x: 1 if (pd.isna(x) or x != 2) else 2)
df_no_na["isna"] = 0

FileNotFoundError: [Errno 2] No such file or directory: 'proses/Data-SBR-20771-20781.csv'

isna = 0 -> Ditemukan <br>
isna = 1 -> Belum dilakukan search <br>
isna = 2 -> Sudah di-search tapi tidak ditemukan <br>
isna = 3 -> Block by Google 

inside_boundary = 0 -> Di luar poligon kecamatan <br>
inside_boundary = 1 -> Di dalam poligon kecamatan <br>

### Scrapping Loop

In [ ]:
import time

start_time = time.time()

while True:
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-extensions")

    driver = webdriver.Chrome(options=options)

    for i, r in tqdm(df_na.iterrows(), total=len(df_na)):
    
        if r["isna"] not in (1, 3):
            continue
    
        location = r["nama"]
        desa = r["nmdesa"]
    
        result = get_long_lat(driver, location, desa)
    
        df_na.loc[i, [
            "latitude",
            "longitude",
            "nama_google",
            "url",
            "isna",
            "temporary_closed"
        ]] = result
    
        # jika ditemukan
        if result[4] == 0:
    
            lat, lon, nama_google, _, _, _ = result
    
            if lat and lon:
                point = Point(float(lon), float(lat))
                inside = int(point.within(polygon))
            else:
                inside = 0
    
            df_na.loc[i, "inside_boundary"] = inside
    
            similarity = fuzz.token_set_ratio(
                str(r["nama"]).lower(),
                str(nama_google).lower()
            )
    
            df_na.loc[i, "similarity"] = similarity
    
        # jika diblock google
        elif result[4] == 3:
            print("Google block detected, restarting driver...")
            break
    
    driver.quit()
    
    result_df = pd.concat([df_na, df_no_na]).sort_index()
    result_df.to_csv(f'proses/{file_input}', index=False, sep=';')

    # jika sudah tidak ada row yang diblock google
    if not (result_df["isna"] == 3).any():
        print("All rows processed successfully.")
        break
    
end_time = time.time()
elapsed = end_time - start_time

print(f"Total time: {elapsed:.2f} seconds")
print(f"Total time: {elapsed/60:.2f} minutes")
print(f"Average time per row: {elapsed/len(df_na):.2f} seconds")
print(f"Total records: {len(result_df)}")
print(f"Records with coordinates: {len(result_df[result_df['latitude'] != ''])}")
print(f"Records without coordinates: {len(result_df[result_df['latitude'] == ''])}")